Q1. Consider the dataset Sacremento.csv. Ignoring the variable “zip”, consider “price” as the response variable and all other as features.
With K-Fold cross-validation with Grid Search with parameters of your choice, evaluate r2 score with following algorithms:
    1. ElasticNet
    2. Random Forest
    3. XG Boost
    4. LightGBM

Q2. Refer Datasets
    A. Consider the dataset Protein.xlsx. Find best K, for K-Means clustering on the basis of Silhouette Scores
    B. State the number PCAs which provide 80% variation

Q3. Consider the dataset Vehicle.csv. Consider the variable “Class” as response variable. 
    1. With Grid Search for Decision Tree considering parameter values for max_depth as [2,4,6] , min_samples_split  as [2,3,5] and min_samples_leaf as [1,3,5], find best parameter set on the basis of log loss.
    2. With Grid Search for Random Forest considering parameter values for max_features=[2,3,4,5]
    3. With Grid Search for Logistic Regression considering parameter values of solvers

Q4. Consider the time series monthly-beer-production-in-austr.csv. Try the following:
    1. Check whether the series is stationary
    2. ARIMA models
    3. Smoothing models






In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split , KFold , GridSearchCV
from sklearn.preprocessing import OneHotEncoder , StandardScaler
from sklearn.compose import make_column_selector , make_column_transformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

In [2]:
df = pd.read_csv(r"C:\Users\PGCP-AI.STUDENTSDC\Downloads\Sacremento.csv")

In [3]:
df.head()

,city,zip,beds,baths,sqft,type,price,latitude,longitude
0,SACRAMENTO,z95838,2,1.0,836,Residential,59222,38.631913,-121.434879
1,SACRAMENTO,z95823,3,1.0,1167,Residential,68212,38.478902,-121.431028
2,SACRAMENTO,z95815,2,1.0,796,Residential,68880,38.618305,-121.443839
3,SACRAMENTO,z95815,2,1.0,852,Residential,69307,38.616835,-121.439146
4,SACRAMENTO,z95824,2,1.0,797,Residential,81900,38.519470,-121.435768


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 932 entries, 0 to 931
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   city       932 non-null    object 
 1   zip        932 non-null    object 
 2   beds       932 non-null    int64  
 3   baths      932 non-null    float64
 4   sqft       932 non-null    int64  
 5   type       932 non-null    object 
 6   price      932 non-null    int64  
 7   latitude   932 non-null    float64
 8   longitude  932 non-null    float64
dtypes: float64(3), int64(3), object(3)
memory usage: 65.7+ KB


In [28]:
df.isnull().sum()

city         0
beds         0
baths        0
sqft         0
type         0
price        0
latitude     0
longitude    0
dtype: int64

In [5]:
df = df.drop("zip", axis = "columns")

In [7]:
X, y = df.drop("price", axis = "columns") , df["price"]

In [29]:
ohe = OneHotEncoder(sparse_output = False, drop = "first", handle_unknown="ignore")
scl = StandardScaler()
ct = make_column_transformer(
    (ohe , make_column_selector(dtype_include = "object")),
    ("passthrough", make_column_selector(dtype_include = "number")),
    verbose_feature_names_out = False
).set_output(transform="pandas")

In [30]:
kfold = KFold(n_splits=5,shuffle=True,random_state=26)

el = ElasticNet()

pipe = Pipeline([
    ("ct" , ct),
    ("scl" , scl),
    ("el", el)
])

In [31]:
params = {
    "el__l1_ratio" : np.linspace(0.01, 0.9, 10),
    "el__alpha" : np.linspace(0.01, 0.9, 10)
}

In [32]:
grid = GridSearchCV(pipe, param_grid = params, cv = kfold, scoring = "r2", n_jobs = -1)

In [33]:
grid.fit(X, y)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...lasticNet())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'el__alpha': array([0.01 ..., 0.9 ]), 'el__l1_ratio': array([0.01 ..., 0.9 ])}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int,

In [34]:
grid.best_score_

np.float64(0.6607048651266769)

RANDOM FOREST

In [36]:
rf = RandomForestRegressor(random_state=42)

In [51]:
pipe = Pipeline([
    ("ct" , ct),
    ("rf", rf)
])

In [53]:
params = {
 
    "rf__max_depth" : [2, 3, 4, 5, 6, 7],
    "rf__n_estimators" : [20, 30, 40, 50],
    "rf__max_samples" : [50, 70, 100, 150, 200]
}

grid = GridSearchCV(pipe, param_grid = params, cv = kfold, scoring = "r2", n_jobs = -1)
grid.fit(X, y)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'rf__max_depth': [2, 3, ...], 'rf__max_samples': [50, 70, ...], 'rf__n_estimators': [20, 30, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"v

In [54]:
grid.best_score_

np.float64(0.6578387418778683)

XGBOOST

In [42]:
xgbr = XGBRegressor(random_state=42)

In [46]:
pipe = Pipeline([
    ("ct" , ct),
    ("xgbr", xgbr)
])


In [56]:
params = {
    "xgbr__max_depth" : [2, 3, 4, 5, 6, 7],
    "xgbr__n_estimators" : [20, 30, 40, 50],
    "xgbr__max_samples" : [50, 70, 100, 150, 200]
}

grid = GridSearchCV(pipe, param_grid = params, cv = kfold, scoring = "r2", n_jobs = -1)
grid.fit(X, y)

ValueError: Invalid parameter 'xgbr' for estimator Pipeline(steps=[('ct',
                 ColumnTransformer(transformers=[('onehotencoder',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x000002205A9AB380>),
                                                 ('passthrough', 'passthrough',
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x0000022056EAA490>)],
                                   verbose_feature_names_out=False)),
                ('rf', RandomForestRegressor(random_state=42))]). Valid parameters are: ['memory', 'steps', 'transform_input', 'verbose'].

In [ ]:
grid.best_score_

In [50]:
lgbmr = LGBMRegressor(random_state=42, verbose=-1)

pipe = Pipeline([
    ("ct" , ct),
    ("lgbmr", lgbmr)
])

params = {
    "lgbmr__max_depth" : [2, 3, 4, 5, 6, 7],
    "lgbmr__n_estimators" : [20, 30, 40, 50],
    "lgbmr__max_samples" : [50, 70, 100, 150, 200]
}

grid = GridSearchCV(pipe, param_grid = params, cv = kfold, scoring = "r2", n_jobs = -1)
grid.fit(X, y)

grid.best_score_

np.float64(0.6645733996005043)

LGBMRegressor = 66.4\
RandomForestRegressor = 65.8\
XGBRegressor = 68.1\
ElasticNet = 66